## Advanced Tracing Features
---
This notebook briefly explores other features of Nsight Systems that were not covered in the other labs. The goal is to learn about tracing of Message Passing Interface (MPI), OpenSHMEM, NVSHMEM™, and NVIDIA Collective Communication Library (NCCL), and also learn how to do multi-process profiling.


#### GPU Metrics Sampling

Nsight Systems has a GPU Metrics feature that identifies performance bottlenecks in applications that use a GPU for computation. It uses periodic sampling to gather performance metrics and detailed timing statistics across different GPU hardware units, leveraging specialized hardware to capture this data in a single pass with minimal overhead. These metrics provide an overview of GPU efficiency over time within compute and input/output (I/O) activities:

- I/O throughputs: PCIe, NVLink, and dynamic random access memory (DRAM)
- Streaming Multiprocessor / Shared Processor (SM)  utilization: SMs activity, Tensor Core activity, instructions issued, warp occupancy (including unallocated slots)

These metrics can also help users answer the common questions:

- Is my GPU idle? 
- Is my instruction rate low (possibly I/O bound)?
- Is my GPU full? Sufficient kernel grid size and streams? Are my SMs and warp slots full?
- Can I see GPU Direct Remote Direct Memory Access (RDMA)/Storage or other transfers?
- Am I using TensorCores?
- Am I possibly blocked on I/O, or number of warps, etc?

<img src="images/gmetric-1.png">

You can simply hover the mouse over the rows and inspect the gathered data. For example, the screenshot below shows the *Compute Warps in Flight*, which is simply the ratio of active compute shader warps resident on the SMs to the maximum number of warps per SM, expressed as a percentage.

<img src="images/gmetric-2.png">

You can also get information on the ratio of bytes received or transmitted on the PCIe or NVLink interface, in addition to the DRAM read/write bandwidth. 

<img src="images/gmetric-3.png">
The GPU Metrics feature is controlled by switches that let you select the sampling frequency, the metrics to use, and the GPUs to sample.

- `--gpu-metrics-devices=[all, cuda-visible, none, <index>]` selects GPUs to sample (default is none).

- `--gpu-metrics-set=[<alias>, file:<file name>]` selects metric set to use (default is the 1st suitable from the list).

- `--gpu-metrics-frequency=[10..200000]` selects sampling frequency in Hz (default is 10000). Each metric has an approximate recommended sampling frequency range. To get accurate data, try to choose a frequency closer to the recommended range.

Nsight Systems GPU Metrics requires NVIDIA GPUs with the Turing™ architecture or newer. Learn more about GPU metrics at https://docs.nvidia.com/nsight-systems/UserGuide/index.html#gpu-metric-sampling

To list available GPUs, use:

In [ ]:
!module load nsight-systems/2024.5.1 && srun --partition=primary -n1 --gres=gpu:1 nsys profile --gpu-metrics-devices=help


#### TRACING OF MPI, OpenSHMEM, NVSHMEM AND NCCL

Nsight Systems supports MPI as well as OpenSHMEM tracing. You can record MPI communication parameters and track the MPI communicators. So, if you want to follow the data and see which MPI ranks are communicating with each other and how much data is transferred, you can see this information in the report. 

<img src="images/mpi_comm.png">

In the above screenshot you see the tooltip for an `MPI_Irecv` (bottom right) with its MPI tag, the number of bytes that have been received, the sender of the data and also the communicator. In the case of MPI, you can also trace MPI for Fortran applications. 

Besides MPI and OpenSHMEM, where we intercept the library calls, Nsight Systems can also trace calls into NVSHMEM and NCCL libraries based on NVIDIA Tools Extension SDK (NVTX) explained in previous labs. The result looks similar to what is shown in the above screenshot, with the function names of the respective API and the respective row labels, e.g. NCCL and NVSHMEM rows instead of MPI and UCX as in the above screenshot.

In the screenshot, you see the execution timeline of a super short range of an MPI program which triggers some `MPI_Isends` and `MPI_Irecvs`. For each MPI call you get the communication parameters, and you also get the Unified Communication X (UCX) API calls, which are triggered by the MPI implementation, which in this example is OpenMPI. 


#### UCX API TRACING

The UCX layer is an open-source communication framework that acts as a common library and API for several higher-level communication libraries, for example Open MPI (including its OpenSHMEM implementation) and MPICH. If UCX library trace is selected Nsight Systems will trace the subset of functions of the UC Protocol layer (UCP) that are most likely involved in performance bottlenecks. If OpenSHMEM library trace is selected Nsight Systems will trace the subset of OpenSHMEM API functions that are most likely involved in performance bottlenecks. 

<img src="images/ucx.png">

In the above screenshot, we have both `MPI_Isend` and `MPI_Irecv` calls that trigger UCP API calls (you only see the `MPI_Isend` calls, because the `MPI_Irecv` calls are super short for this particular example). The bottom row in the timeline shows the processing of transfers from non-blocking UCP communication operations. In the UCX row, you see the submit functions and in the row below you see when the processing of the transfers starts (if we were to zoom out, we could also see when the processing ends).

#### NIC Performance Metrics
NVIDIA ConnectX® smart network interface cards (smart NICs) offer advanced hardware offloads and accelerations for network operations. Viewing smart NICs metrics, on Nsight Systems timeline, enables developers to better understand their application’s network usage and use this information to optimize the application’s performance.

<img src="images/NIC.png" width="70%">

The performance counters are displayed over the Nsight Systems timeline, letting you know when the application is sending and receiving data. There are also metrics that indicate network congestion like the `Bytes sent/sec` and the `Bytes received/sec` and help identify idle and busy NIC times. 

One can shift network operations from busy to idle times to reduce network congestion and latency or use idle NIC times to send additional data without reducing application performance.

To collect NIC performance metrics, using Nsight Systems CLI, you would need to add the `--nic-metrics` command line switch when profiling: `nsys profile --nic-metrics=true my_app`


### Nsight Systems Multi-Process Profiling

On compute clusters where you have to use a workload manager or want to do a run over multiple nodes, the `nsys profile` command is prefixed before the application. With this, a report file is generated for each process. If you can launch your application without a workload manager on a single node (e.g. with `mpirun`), you can prefix `nsys profile` before `mpirun` and  a single report including all processes is generated.

- **Single Node**: `nsys profile [nsys_args] mpirun [mpirun_args] your_executable`. The command will create one report file.
- **Multiple Nodes**: `mpirun [mpirun_args] nsys profile [nsys_args] your_executable`, you can set the output report name with `-o report_name_%q{OMPI_COMM_WORLD_RANK}`. (For OpenMPI, PMI_RANK for MPICH and SLURM_PROCID for Slurm). The command will create one report file per MPI rank.
   
You can also profile only specific ranks: 

```
#!/bin/bash
# OMPI_COMM_WORLD_LOCAL_RANK for node local rank
if [ $OMPI_COMM_WORLD_RANK -eq 0 ]; then
    nsys profile -t mpi "$@"
else
    "$@"
fi
```

Below is an example command that will be used to profile the ddp model on multiple nodes (on a cluster that uses *SLURM* workload manager).

```
srun [SRUN_ARGS] nsys profile --trace cuda,osrt,nvtx,ucx \
--gpu-metrics-device=cuda-visible \
--gpu-metrics-frequency=5000 \
--nic-metrics=true \
--output reports/output%q{SLURM_NODEID}_%q{SLURM_PROCID} \
--force-overwrite true ./myprogram [PROGRAM_ARGS]
```

where,

- `nsys profile`: starts a profiling session
- `-t, --trace=...`: sets the APIs to be traced, in this example, it is UCX and MPI
- `-s,--sample=[cpu|none]`: controls CPU IP sampling
- `gpu-metrics-device=cuda-visible`:  Select GPUs that match CUDA_VISIBLE_DEVICES
- `--nic-metrics=[true|false]`: controls Network Interface Cards (NIC) metrics collection
- `-o, --output=report#`: output profile report file path; and
- `-f --force-overwrite`: overwrite the output report file, if it already exists

To learn more about other switches to use with `nsys profile`, you can type `nsys profile --help` on the command line or read the [documentation](https://docs.nvidia.com/nsight-systems/UserGuide/index.html#cli-profiling).


### Multi-Report Analysis

Nsight Systems Multi-Report Analysis is a functionality to better support complex statistical analysis across multiple result files, including Multi-Node Analysis that will be used throughout the following sections. When you run Nsight Systems across a cluster, it typically generates one result file per rank on the cluster. While you can load multiple result files into the GUI for visualization, this analysis system allows you to run statistical analysis across all of the result files.

There are multi-report recipes that can be used to get a deeper analysis of the metrics and starts. Please note that all recipes are in the form of Python scripts, and one can alter the given recipes or write their own. To learn more about available built-in recipes,  use `nsys recipe --help`, or read the [documentation](https://docs.nvidia.com/nsight-systems/UserGuide/index.html?highlight=node#multi-report-analysis).

Please click on the `Next Notebook` link below to proceed to the multinode lab.

## <div style="text-align: center ;border:3px; border-style:solid; border-color:#FF0000  ; padding: 1em">[Next Notebook](multinode.ipynb)</div>



# Links and Resources


[NVIDIA Nsight System](https://docs.nvidia.com/nsight-systems/)


**NOTE**: To be able to see the Nsight System Profiler outputs, please download the latest versions from the pages:

- https://developer.nvidia.com/nsight-systems

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

--- 

## Licensing 

Copyright © 2022 OpenACC-Standard.org.  This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.